# 🧬 Attention Gate Comparison Study
### Soft Attention vs No Attention vs Sparse Attention
**Run order:**
1. Cell 1 — Recovery
2. Cell 2 — No-gate ablation (1 seed × all 5 cancers + 5×5 KIRC)
3. Cell 3 — Sparse-gate ablation (1 seed × all 5 cancers + 5×5 KIRC)
4. Cell 4 — Three-way comparison figures

**Design:**
- All 5 cancers, all 63 combos, seed=42 only (315 runs per gate type)
- KIRC: additionally runs all 5 seeds × 5 folds (1,575 runs per gate type)
- Soft attention results already exist in phase1_results/ — reused here


In [ ]:
# ── CELL 1: Recovery ─────────────────────────────────────────────────────────
import os, time, pickle, json, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from itertools import combinations
from scipy import stats as scipy_stats
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE_DIR   = Path('/content/drive/MyDrive/tcga_pancancer')
PROC_DIR   = BASE_DIR / 'processed'
MDL_DIR    = BASE_DIR / 'models'
RES_DIR    = BASE_DIR / 'results'
SUB_DIR    = PROC_DIR / 'subtypes'
PHASE1_DIR = SUB_DIR / 'phase1_results'       # existing soft-gate results
NOGATE_DIR = SUB_DIR / 'nogate_results'        # new: no gate
SPARSE_DIR = SUB_DIR / 'sparse_results'        # new: sparse gate
for d in [PHASE1_DIR, NOGATE_DIR, SPARSE_DIR]:
    d.mkdir(exist_ok=True)

FIG_DIR = RES_DIR / 'attention_comparison_figures'
FIG_DIR.mkdir(exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
TARGET_CANCERS = ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']
LATENT_DIMS    = {'rna':256,'meth':256,'cnv':128,'mirna':128,'mut':128,'clin':64}
HIDDEN_CFG     = {
    'rna':   [2048,1024,512], 'meth':  [2048,1024,512],
    'cnv':   [256,128],       'mirna': [512,256],
    'mut':   [512,256],       'clin':  [64,64],
}

print("Loading modality matrices...")
t0 = time.time()
mod_data, mod_maps, mod_dims = [], [], {}
for mod in MODALITY_NAMES:
    df = pd.read_parquet(PROC_DIR / f'{mod}_matrix.parquet')
    mod_maps.append({cid: i for i, cid in enumerate(df.index)})
    mod_data.append(df.values.astype(np.float32))
    mod_dims[mod] = df.shape[1]
    print(f"  {mod}: {df.shape}")
print(f"All matrices loaded in {time.time()-t0:.1f}s")

print("\nLoading subtype datasets...")
with open(SUB_DIR / 'subtype_datasets.pkl', 'rb') as f:
    subtype_datasets = pickle.load(f)
for cancer, ds in subtype_datasets.items():
    print(f"  {cancer}: {len(ds['df'])} samples, {ds['n_subtypes']} subtypes")

if (SUB_DIR / 'ablation_results.pkl').exists():
    with open(SUB_DIR / 'ablation_results.pkl', 'rb') as f:
        ablation_results = pickle.load(f)
    print(f"\nSingle-seed ablation results: {sum(len(v) for v in ablation_results.values())} entries")

# ── Dataset + collate ─────────────────────────────────────────────────────────
class SubtypeDataset(Dataset):
    def __init__(self, rows, labels, mod_data, mod_maps, case_ids_for_rows):
        self.rows=rows; self.labels=labels
        self.mod_data=mod_data; self.mod_maps=mod_maps; self.cids=case_ids_for_rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        cid = self.cids[i]
        tensors, mask = [], []
        for data, mapping in zip(self.mod_data, self.mod_maps):
            if cid in mapping:
                tensors.append(torch.tensor(data[mapping[cid]], dtype=torch.float32))
                mask.append(1.0)
            else:
                tensors.append(torch.zeros(data.shape[1], dtype=torch.float32))
                mask.append(0.0)
        return tensors, torch.tensor(mask, dtype=torch.float32), \
               torch.tensor(self.labels[i], dtype=torch.long)

def collate_fn(batch):
    tensors_list, masks, labels = zip(*batch)
    return ([torch.stack([s[m] for s in tensors_list]) for m in range(len(tensors_list[0]))],
            torch.stack(masks), torch.stack(labels))

def make_encoder(in_dim, latent_dim, hidden):
    layers, prev = [], in_dim
    for h in hidden:
        layers += [nn.Linear(prev,h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2)]
        prev = h
    layers.append(nn.Linear(prev, latent_dim))
    return nn.Sequential(*layers)

# ── Three model variants ──────────────────────────────────────────────────────
class NoGateModel(nn.Module):
    """Encoders → simple concatenation → classifier. No attention gate at all."""
    def __init__(self, active_indices, mod_dims, n_classes):
        super().__init__()
        self.active   = list(active_indices)
        self.encoders = nn.ModuleList([
            make_encoder(mod_dims[MODALITY_NAMES[i]], LATENT_DIMS[MODALITY_NAMES[i]],
                         HIDDEN_CFG[MODALITY_NAMES[i]]) for i in self.active])
        fusion_dim = sum(LATENT_DIMS[MODALITY_NAMES[i]] for i in self.active)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim,256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256,128),        nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes))
    def forward(self, tensors, mask):
        zs    = [enc(tensors[i]) for enc, i in zip(self.encoders, self.active)]
        fused = torch.cat(zs, dim=-1)
        return self.classifier(fused)

class SparseGateModel(nn.Module):
    """Encoders → sparsemax attention gate → classifier.
    Sparsemax produces truly sparse weights that can be exactly 0,
    unlike sigmoid which only approaches 0 asymptotically.
    Reference: Martins & Astudillo (2016) arXiv:1602.02068
    """
    def __init__(self, active_indices, mod_dims, n_classes):
        super().__init__()
        self.active   = list(active_indices)
        self.n_active = len(self.active)
        self.encoders = nn.ModuleList([
            make_encoder(mod_dims[MODALITY_NAMES[i]], LATENT_DIMS[MODALITY_NAMES[i]],
                         HIDDEN_CFG[MODALITY_NAMES[i]]) for i in self.active])
        # Gate: projects mask availability to sparse attention weights
        self.gate_proj = nn.Linear(self.n_active, self.n_active)
        fusion_dim = sum(LATENT_DIMS[MODALITY_NAMES[i]] for i in self.active)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim,256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256,128),        nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes))

    @staticmethod
    def sparsemax(z):
        """Sparsemax activation (Martins & Astudillo 2016).
        Projects onto the probability simplex — outputs are non-negative,
        sum to 1, and can be exactly zero (unlike softmax/sigmoid).
        """
        z_sorted, _ = torch.sort(z, dim=-1, descending=True)
        k = torch.arange(1, z.shape[-1]+1, dtype=z.dtype, device=z.device)
        z_cumsum = torch.cumsum(z_sorted, dim=-1)
        rho = ((z_sorted * k) > (z_cumsum - 1)).sum(dim=-1, keepdim=True)
        theta = (z_cumsum.gather(-1, rho-1) - 1) / rho.float()
        return torch.clamp(z - theta, min=0)

    def forward(self, tensors, mask):
        active_tensors = [tensors[i] for i in self.active]
        active_mask    = mask[:, self.active]
        zs   = [enc(x) for enc, x in zip(self.encoders, active_tensors)]
        # Sparse gate: learn which modalities to zero out
        gate = self.sparsemax(self.gate_proj(active_mask))   # sums to 1, can be 0
        fused = torch.cat([z * gate[:, i:i+1] for i, z in enumerate(zs)], dim=-1)
        return self.classifier(fused)

class SoftGateModel(nn.Module):
    """Original soft sigmoid gate — included for completeness / symmetry."""
    def __init__(self, active_indices, mod_dims, n_classes):
        super().__init__()
        self.active   = list(active_indices)
        self.encoders = nn.ModuleList([
            make_encoder(mod_dims[MODALITY_NAMES[i]], LATENT_DIMS[MODALITY_NAMES[i]],
                         HIDDEN_CFG[MODALITY_NAMES[i]]) for i in self.active])
        self.gate       = nn.Linear(len(self.active), len(self.active))
        fusion_dim      = sum(LATENT_DIMS[MODALITY_NAMES[i]] for i in self.active)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim,256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256,128),        nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, n_classes))
    def forward(self, tensors, mask):
        active_tensors = [tensors[i] for i in self.active]
        active_mask    = mask[:, self.active]
        zs    = [enc(x) for enc, x in zip(self.encoders, active_tensors)]
        gate  = torch.sigmoid(self.gate(active_mask))
        fused = torch.cat([z * gate[:, i:i+1] for i, z in enumerate(zs)], dim=-1)
        return self.classifier(fused)

print("\n✅ Recovery complete")
print(f"   Models defined: NoGateModel, SparseGateModel, SoftGateModel")
print(f"   NOGATE_DIR: {NOGATE_DIR}")
print(f"   SPARSE_DIR: {SPARSE_DIR}")
nogate_done = len(list(NOGATE_DIR.glob('*.json')))
sparse_done = len(list(SPARSE_DIR.glob('*.json')))
print(f"   No-gate runs completed: {nogate_done}")
print(f"   Sparse-gate runs completed: {sparse_done}")


Mounted at /content/drive
Device: cuda
Loading modality matrices...
  rna: (11505, 5000)
  meth: (12527, 10000)
  cnv: (15000, 41)
  mirna: (11441, 508)
  mut: (9106, 501)
  clin: (11428, 7)
All matrices loaded in 65.4s

Loading subtype datasets...
  TCGA-BRCA: 1216 samples, 5 subtypes
  TCGA-GBM_LGG: 723 samples, 7 subtypes
  TCGA-UCEC: 507 samples, 4 subtypes
  TCGA-THCA: 482 samples, 5 subtypes
  TCGA-KIRC: 417 samples, 4 subtypes

Single-seed ablation results: 315 entries

✅ Recovery complete
   Models defined: NoGateModel, SparseGateModel, SoftGateModel
   NOGATE_DIR: /content/drive/MyDrive/tcga_pancancer/processed/subtypes/nogate_results
   SPARSE_DIR: /content/drive/MyDrive/tcga_pancancer/processed/subtypes/sparse_results
   No-gate runs completed: 1827
   Sparse-gate runs completed: 1827


In [ ]:
# ── CELL 2: No-Gate Ablation ─────────────────────────────────────────────────
# All 5 cancers × 63 combos × seed=42 = 315 runs  (~2 hrs on A100)
# KIRC additionally: 5 seeds × 5 folds = 1,575 runs total for KIRC  (~9 hrs)
# Resume-safe

SEEDS_KIRC   = [42, 123, 456, 789, 2026]
N_FOLDS_KIRC = 5
SEEDS_OTHER  = [42]   # single seed for non-KIRC cancers
N_FOLDS_OTHER= 1
EPOCHS = 60
LR     = 1e-3

def get_fold_indices(y_all, seed, fold, n_folds=5):
    y_arr = np.array(y_all)
    if n_folds == 1:
        # Single split: 70/15/15
        idx = np.arange(len(y_arr))
        idx_tv, test_idx = train_test_split(idx, test_size=0.15, stratify=y_arr,
                                            random_state=seed)
        train_idx, val_idx = train_test_split(idx_tv, test_size=0.15/0.85,
                                              stratify=y_arr[idx_tv], random_state=seed)
        return train_idx, val_idx, test_idx
    skf    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    splits = list(skf.split(np.zeros(len(y_arr)), y_arr))
    train_val_idx, test_idx = splits[fold]
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.15/0.85,
                                          stratify=y_arr[train_val_idx], random_state=seed)
    return train_idx, val_idx, test_idx

def result_path(save_dir, cancer, combo, seed, fold):
    combo_str = '+'.join(MODALITY_NAMES[i] for i in combo)
    return save_dir / f"{cancer.replace('TCGA-','')}_{combo_str}_s{seed}_f{fold}.json"

def train_one_run(ModelClass, cancer, combo, seed, fold, n_folds, save_dir):
    """Train one model run. Returns result dict and saves JSON."""
    rpath = result_path(save_dir, cancer, combo, seed, fold)
    if rpath.exists():
        return None   # already done

    torch.manual_seed(seed); np.random.seed(seed)
    ds    = subtype_datasets[cancer]
    df    = ds['df']
    n_sub = ds['n_subtypes']
    y_all = df['subtype_label'].values
    cids  = df['uuid'].values

    train_idx, val_idx, test_idx = get_fold_indices(y_all.tolist(), seed, fold, n_folds)

    def make_loader(idx, weighted=False):
        labels  = y_all[idx]
        dataset = SubtypeDataset(idx, labels, mod_data, mod_maps, cids[idx])
        sampler, shuffle = None, True
        if weighted:
            counts  = np.bincount(labels, minlength=n_sub)
            counts  = np.where(counts==0,1,counts)
            weights = 1.0/counts[labels]
            sampler = WeightedRandomSampler(weights, len(weights))
            shuffle = False
        return DataLoader(dataset, batch_size=32, shuffle=shuffle,
                          sampler=sampler, collate_fn=collate_fn)

    train_loader = make_loader(train_idx, weighted=True)
    val_loader   = make_loader(val_idx)
    test_loader  = make_loader(test_idx)

    model = ModelClass(list(combo), mod_dims, n_sub).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    best_val_f1, best_state = 0.0, None
    for epoch in range(EPOCHS):
        model.train()
        for tensors, mask, labels in train_loader:
            tensors=[t.to(DEVICE) for t in tensors]
            mask,labels=mask.to(DEVICE),labels.to(DEVICE)
            opt.zero_grad()
            nn.CrossEntropyLoss()(model(tensors,mask),labels).backward()
            opt.step()
        sched.step()
        model.eval()
        preds,trues=[],[]
        with torch.no_grad():
            for tensors,mask,labels in val_loader:
                tensors=[t.to(DEVICE) for t in tensors]
                preds.extend(model(tensors,mask.to(DEVICE)).argmax(1).cpu().numpy())
                trues.extend(labels.numpy())
        val_f1 = f1_score(trues,preds,average='macro',zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1=val_f1
            best_state=copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state); model.eval()
    preds,trues=[],[]
    with torch.no_grad():
        for tensors,mask,labels in test_loader:
            tensors=[t.to(DEVICE) for t in tensors]
            preds.extend(model(tensors,mask.to(DEVICE)).argmax(1).cpu().numpy())
            trues.extend(labels.numpy())

    test_f1  = float(f1_score(trues,preds,average='macro',zero_division=0))
    test_acc = float(np.mean(np.array(preds)==np.array(trues)))
    result   = {
        'cancer': cancer, 'combo': list(combo),
        'combo_name': '+'.join(MODALITY_NAMES[i] for i in combo),
        'seed':seed, 'fold':fold, 'n_folds':n_folds,
        'test_f1':test_f1, 'test_acc':test_acc, 'best_val_f1':float(best_val_f1),
        'n_train':int(len(train_idx)), 'n_val':int(len(val_idx)), 'n_test':int(len(test_idx)),
        'model_type': ModelClass.__name__,
    }
    with open(rpath,'w') as f:
        json.dump(result,f)
    return result

# ── Build run list for no-gate ────────────────────────────────────────────────
all_combos = []
for r in range(1,7):
    all_combos.extend(combinations(range(6),r))

run_list = []
for cancer in TARGET_CANCERS:
    seeds  = SEEDS_KIRC  if cancer=='TCGA-KIRC' else SEEDS_OTHER
    n_fold = N_FOLDS_KIRC if cancer=='TCGA-KIRC' else N_FOLDS_OTHER
    for combo in all_combos:
        for seed in seeds:
            folds = range(n_fold) if n_fold > 1 else [0]
            for fold in folds:
                run_list.append((cancer, combo, seed, fold, n_fold))

total     = len(run_list)
completed = sum(1 for (c,co,s,f,nf) in run_list
                if result_path(NOGATE_DIR,c,co,s,f).exists())
print(f"No-gate run plan:")
print(f"  Non-KIRC: {4*63} runs (1 seed)")
print(f"  KIRC: {63*5*5} runs (5 seeds × 5 folds)")
print(f"  Total: {total}")
print(f"  Completed: {completed}")
print(f"  Remaining: {total-completed}")
print(f"  Est. time on A100: {(total-completed)*21/3600:.1f} hrs\n")

run_count = 0
t_session = time.time()

for cancer, combo, seed, fold, n_folds in run_list:
    rpath = result_path(NOGATE_DIR, cancer, combo, seed, fold)
    if rpath.exists():
        continue
    t0     = time.time()
    result = train_one_run(NoGateModel, cancer, combo, seed, fold, n_folds, NOGATE_DIR)
    if result is None:
        continue
    run_count += 1
    completed += 1
    elapsed = time.time()-t0
    print(f"[{completed}/{total}] {cancer.replace('TCGA-','')} "
          f"{result['combo_name']:<30} s{seed} f{fold} "
          f"F1={result['test_f1']:.4f} ({elapsed:.0f}s) session={run_count}")

print(f"\n✅ No-gate session complete. {run_count} new runs.")


No-gate run plan:
  Non-KIRC: 252 runs (1 seed)
  KIRC: 1575 runs (5 seeds × 5 folds)
  Total: 1827
  Completed: 1827
  Remaining: 0
  Est. time on A100: 0.0 hrs


✅ No-gate session complete. 0 new runs.


In [ ]:
# ── CELL 3: Sparse-Gate Ablation ─────────────────────────────────────────────
# Identical run plan to Cell 2 but using SparseGateModel
# Resume-safe — can run immediately after Cell 2 or in a separate session

all_combos = []
for r in range(1,7):
    all_combos.extend(combinations(range(6),r))

run_list = []
for cancer in TARGET_CANCERS:
    seeds  = SEEDS_KIRC  if cancer=='TCGA-KIRC' else SEEDS_OTHER
    n_fold = N_FOLDS_KIRC if cancer=='TCGA-KIRC' else N_FOLDS_OTHER
    for combo in all_combos:
        for seed in seeds:
            folds = range(n_fold) if n_fold > 1 else [0]
            for fold in folds:
                run_list.append((cancer, combo, seed, fold, n_fold))

total     = len(run_list)
completed = sum(1 for (c,co,s,f,nf) in run_list
                if result_path(SPARSE_DIR,c,co,s,f).exists())
print(f"Sparse-gate run plan:")
print(f"  Total: {total}")
print(f"  Completed: {completed}")
print(f"  Remaining: {total-completed}")
print(f"  Est. time on A100: {(total-completed)*22/3600:.1f} hrs\n")

run_count = 0

for cancer, combo, seed, fold, n_folds in run_list:
    rpath = result_path(SPARSE_DIR, cancer, combo, seed, fold)
    if rpath.exists():
        continue
    t0     = time.time()
    result = train_one_run(SparseGateModel, cancer, combo, seed, fold, n_folds, SPARSE_DIR)
    if result is None:
        continue
    run_count += 1
    completed += 1
    elapsed = time.time()-t0
    print(f"[{completed}/{total}] {cancer.replace('TCGA-','')} "
          f"{result['combo_name']:<30} s{seed} f{fold} "
          f"F1={result['test_f1']:.4f} ({elapsed:.0f}s) session={run_count}")

print(f"\n✅ Sparse-gate session complete. {run_count} new runs.")


Sparse-gate run plan:
  Total: 1827
  Completed: 1488
  Remaining: 339
  Est. time on A100: 2.1 hrs

[1489/1827] KIRC rna+cnv+mut+clin               s456 f1 F1=0.6075 (11s) session=1
[1490/1827] KIRC rna+cnv+mut+clin               s456 f2 F1=0.4915 (11s) session=2
[1491/1827] KIRC rna+cnv+mut+clin               s456 f3 F1=0.5353 (11s) session=3
[1492/1827] KIRC rna+cnv+mut+clin               s456 f4 F1=0.5413 (11s) session=4
[1493/1827] KIRC rna+cnv+mut+clin               s789 f0 F1=0.2787 (11s) session=5
[1494/1827] KIRC rna+cnv+mut+clin               s789 f1 F1=0.2327 (11s) session=6
[1495/1827] KIRC rna+cnv+mut+clin               s789 f2 F1=0.2298 (11s) session=7
[1496/1827] KIRC rna+cnv+mut+clin               s789 f3 F1=0.2515 (11s) session=8
[1497/1827] KIRC rna+cnv+mut+clin               s789 f4 F1=0.1927 (11s) session=9
[1498/1827] KIRC rna+cnv+mut+clin               s2026 f0 F1=0.6663 (11s) session=10
[1499/1827] KIRC rna+cnv+mut+clin               s2026 f1 F1=0.7790 (11s) sess

In [ ]:
# ── CELL 4: Three-Way Comparison Figures ────────────────────────────────────
# Loads results from all three gate types and produces comparison figures.
# Can be run partially if only one new gate type is complete.

C = {'navy':'#0D1F3C','teal':'#0E7C7B','accent':'#F59E0B',
     'red':'#C0392B','green':'#27AE60','gray':'#64748B','lgray':'#CBD5E1'}
GATE_STYLES = {
    'soft':   {'color':'#2E86AB', 'label':'Soft gate (sigmoid)', 'ls':'-',  'marker':'o'},
    'none':   {'color':'#F59E0B', 'label':'No gate (concat)',    'ls':'--', 'marker':'s'},
    'sparse': {'color':'#C73E1D', 'label':'Sparse gate (sparsemax)','ls':'-.','marker':'^'},
}
plt.rcParams.update({'font.family':'sans-serif','font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
full_6 = '+'.join(MODALITY_NAMES)

def load_results(save_dir, seed_filter=None):
    """Load all JSONs from a directory, optionally filtering by seed."""
    rows = []
    for f in save_dir.glob('*.json'):
        with open(f) as fp:
            r = json.load(fp)
        if seed_filter is None or r['seed'] in seed_filter:
            rows.append(r)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

def best_mean_f1(df, cancer, combo_name):
    """Mean F1 across all runs for a given (cancer, combo)."""
    sub = df[(df['cancer']==cancer) & (df['combo_name']==combo_name)]
    return (sub['test_f1'].mean(), sub['test_f1'].std()) if len(sub)>0 else (np.nan,np.nan)

# Load all three datasets
print("Loading results...")
df_soft   = load_results(PHASE1_DIR, seed_filter=[42])    # single-seed soft from phase1
df_nogate = load_results(NOGATE_DIR, seed_filter=[42])
df_sparse = load_results(SPARSE_DIR, seed_filter=[42])

# For KIRC, use multi-seed results
df_soft_kirc   = load_results(PHASE1_DIR)
df_nogate_kirc = load_results(NOGATE_DIR)
df_sparse_kirc = load_results(SPARSE_DIR)

print(f"  Soft (seed=42):   {len(df_soft)} runs")
print(f"  No-gate (seed=42):{len(df_nogate)} runs")
print(f"  Sparse (seed=42): {len(df_sparse)} runs")

all_combos = []
for r in range(1,7):
    all_combos.extend(combinations(range(6),r))
combo_names = ['+'.join(MODALITY_NAMES[i] for i in c) for c in all_combos]

# ── Figure 1: Best F1 per cancer — three gate types side by side ──────────────
fig, axes = plt.subplots(1, 5, figsize=(20,5), sharey=False)
fig.suptitle('Best Combination F1 by Gate Type Across Cancers\n(seed=42)',
             fontweight='bold', fontsize=14, color=C['navy'])

for ax, cancer in zip(axes, TARGET_CANCERS):
    gate_bests = {}
    for gate_name, df in [('soft',df_soft),('none',df_nogate),('sparse',df_sparse)]:
        if df.empty: continue
        sub = df[df['cancer']==cancer]
        if sub.empty: continue
        best_f1 = sub.groupby('combo_name')['test_f1'].mean().max()
        best_combo = sub.groupby('combo_name')['test_f1'].mean().idxmax()
        gate_bests[gate_name] = (best_f1, best_combo)

    x      = np.arange(len(gate_bests))
    labels = list(gate_bests.keys())
    vals   = [gate_bests[g][0] for g in labels]
    colors = [GATE_STYLES[g]['color'] for g in labels]
    bars   = ax.bar(x, vals, color=colors, edgecolor='white', width=0.6)
    for bar, val, g in zip(bars, vals, labels):
        combo = gate_bests[g][1]
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}\n{combo}', ha='center', va='bottom',
                fontsize=7, color=C['navy'])
    ax.set_xticks(x)
    ax.set_xticklabels([GATE_STYLES[g]['label'].split(' ')[0] for g in labels],
                       fontsize=9)
    ax.set_title(cancer.replace('TCGA-',''), fontweight='bold', fontsize=12,
                 color=C['navy'])
    ax.set_ylim(0,1.1)
    if ax == axes[0]:
        ax.set_ylabel('Macro F1', fontsize=11)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR/'fig1_best_per_cancer.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig1_best_per_cancer.png")

# ── Figure 2: Full model F1 comparison — the key test of gate quality ─────────
fig, ax = plt.subplots(figsize=(9,5))
cancers_short = [c.replace('TCGA-','') for c in TARGET_CANCERS]
x = np.arange(len(TARGET_CANCERS))
width = 0.26

for i, (gate_name, df) in enumerate([('soft',df_soft),
                                       ('none',df_nogate),
                                       ('sparse',df_sparse)]):
    if df.empty: continue
    vals = []
    for cancer in TARGET_CANCERS:
        sub = df[(df['cancer']==cancer) & (df['combo_name']==full_6)]
        vals.append(sub['test_f1'].mean() if len(sub)>0 else np.nan)
    st = GATE_STYLES[gate_name]
    ax.bar(x + i*width - width, vals, width=width,
           color=st['color'], label=st['label'], edgecolor='white', alpha=0.9)
    for xi, val in zip(x, vals):
        if not np.isnan(val):
            ax.text(xi + i*width - width, val+0.01, f'{val:.3f}',
                    ha='center', fontsize=8, color=C['navy'])

ax.set_xticks(x)
ax.set_xticklabels(cancers_short)
ax.set_ylabel('Macro F1 (full 6-modality model)', fontsize=11)
ax.set_title('Full Model Performance by Gate Type\n(Higher = gate handles noise better)',
             fontweight='bold', fontsize=13, color=C['navy'])
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR/'fig2_full_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig2_full_model_comparison.png")

# ── Figure 3: KIRC multi-seed — best combo F1 distributions (boxplot) ─────────
fig, axes = plt.subplots(1,3, figsize=(14,5), sharey=True)
fig.suptitle('KIRC: F1 Distribution for Best Combination\n(5 seeds × 5 folds = 25 runs each)',
             fontweight='bold', fontsize=13, color=C['navy'])

for ax, (gate_name, df_kirc) in zip(axes, [
        ('soft',   df_soft_kirc),
        ('none',   df_nogate_kirc),
        ('sparse', df_sparse_kirc)]):
    if df_kirc.empty:
        ax.text(0.5,0.5,'No data yet', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color=C['gray'])
        ax.set_title(GATE_STYLES[gate_name]['label'])
        continue
    kirc = df_kirc[df_kirc['cancer']=='TCGA-KIRC']
    if kirc.empty:
        ax.text(0.5,0.5,'No KIRC data', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        continue
    # Get top 5 combos by mean
    top5 = kirc.groupby('combo_name')['test_f1'].mean().nlargest(5).index.tolist()
    data = [kirc[kirc['combo_name']==c]['test_f1'].values for c in top5]
    bp   = ax.boxplot(data, patch_artist=True, notch=False,
                      medianprops={'color':'black','linewidth':2})
    for patch in bp['boxes']:
        patch.set_facecolor(GATE_STYLES[gate_name]['color'])
        patch.set_alpha(0.7)
    short_names = [n.replace('rna','r').replace('meth','m').replace('clin','cl')
                   .replace('mirna','mi').replace('cnv','c').replace('mut','mu')
                   for n in top5]
    ax.set_xticklabels(short_names, fontsize=8, rotation=20, ha='right')
    ax.set_title(f"{GATE_STYLES[gate_name]['label']}\n"
                 f"Best: {top5[0]} ({kirc[kirc['combo_name']==top5[0]]['test_f1'].mean():.3f})",
                 fontsize=10, color=C['navy'])
    ax.set_ylabel('Macro F1' if ax==axes[0] else '', fontsize=11)
    ax.set_ylim(0.3,1.0)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(0.5, color=C['gray'], linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR/'fig3_kirc_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig3_kirc_distributions.png")

# ── Figure 4: KIRC best-per-n-modalities across gate types ───────────────────
fig, ax = plt.subplots(figsize=(9,5))
for gate_name, df_kirc in [('soft',df_soft_kirc),
                             ('none',df_nogate_kirc),
                             ('sparse',df_sparse_kirc)]:
    if df_kirc.empty: continue
    kirc = df_kirc[df_kirc['cancer']=='TCGA-KIRC'].copy()
    if kirc.empty: continue
    kirc['n_mod'] = kirc['combo_name'].apply(lambda x: len(x.split('+')))
    best_per_n = kirc.groupby('n_mod')['test_f1'].mean()
    st = GATE_STYLES[gate_name]
    ax.plot(best_per_n.index, best_per_n.values,
            color=st['color'], linestyle=st['ls'],
            marker=st['marker'], markersize=8, linewidth=2,
            label=st['label'], markerfacecolor='white', markeredgewidth=2)

ax.set_xlabel('Number of Modalities', fontsize=12)
ax.set_ylabel('Best Mean Macro F1', fontsize=12)
ax.set_title('KIRC: Best Performance by Modality Count\nThree Gate Types Compared',
             fontweight='bold', fontsize=13, color=C['navy'])
ax.set_xticks(range(1,7))
ax.set_ylim(0.3,1.0)
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR/'fig4_kirc_by_n_gate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig4_kirc_by_n_gate_comparison.png")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "="*80)
print(f"{'Cancer':<14} {'Gate':<10} {'Best Combo':<30} {'Best F1':>8} {'Full-6 F1':>10} {'Delta':>8}")
print("-"*80)
for cancer in TARGET_CANCERS:
    for gate_name, df in [('soft',df_soft),('none',df_nogate),('sparse',df_sparse)]:
        if df.empty: continue
        sub = df[df['cancer']==cancer]
        if sub.empty: continue
        grp      = sub.groupby('combo_name')['test_f1'].mean()
        best_f1  = grp.max()
        best_c   = grp.idxmax()
        full_f1  = grp.get(full_6, np.nan)
        delta    = best_f1 - full_f1
        print(f"{cancer.replace('TCGA-',''):<14} {gate_name:<10} {best_c:<30} "
              f"{best_f1:>8.4f} {full_f1:>10.4f} {delta:>+8.4f}")
    print()


Loading results...
  Soft (seed=42):   894 runs
  No-gate (seed=42):567 runs
  Sparse (seed=42): 567 runs
Saved fig1_best_per_cancer.png
Saved fig2_full_model_comparison.png
Saved fig3_kirc_distributions.png
Saved fig4_kirc_by_n_gate_comparison.png

Cancer         Gate       Best Combo                      Best F1  Full-6 F1    Delta
--------------------------------------------------------------------------------
BRCA           soft       rna+mut+clin                     0.7639     0.7043  +0.0596
BRCA           none       rna+meth+cnv+mut+clin            0.7882     0.6841  +0.1041
BRCA           sparse     rna+mut                          0.7691     0.6706  +0.0985

GBM_LGG        soft       rna+meth+mirna                   0.9310        nan     +nan
GBM_LGG        none       meth+cnv+mut                     0.8934     0.7639  +0.1295
GBM_LGG        sparse     rna+meth+cnv+mirna+mut+clin      0.8884     0.8884  +0.0000

UCEC           none       rna+meth+cnv+mut                 0.7919

In [ ]:
# ── Regenerate fig2 with soft gate filled in from ablation_results.pkl ────────
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations

MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
TARGET_CANCERS = ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']
full_6 = '+'.join(MODALITY_NAMES)

C = {'navy':'#0D1F3C','teal':'#0E7C7B','accent':'#F59E0B',
     'red':'#C0392B','green':'#27AE60','gray':'#64748B','lgray':'#CBD5E1'}

GATE_STYLES = {
    'soft':   {'color':'#2E86AB', 'label':'Soft gate (sigmoid)'},
    'none':   {'color':'#F59E0B', 'label':'No gate (concat)'},
    'sparse': {'color':'#C73E1D', 'label':'Sparse gate (sparsemax)'},
}

# ── Load data ─────────────────────────────────────────────────────────────────

# Soft gate: pull from ablation_results.pkl (single-seed, same as other gates)
with open(SUB_DIR / 'ablation_results.pkl', 'rb') as f:
    ablation_results = pickle.load(f)

# No-gate and sparse-gate: load from JSON dirs (seed=42 only)
def load_results_seed42(save_dir):
    rows = []
    for f in save_dir.glob('*.json'):
        with open(f) as fp:
            r = json.load(fp)
        if r['seed'] == 42 and r.get('fold', 0) == 0:
            rows.append(r)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

df_nogate = load_results_seed42(NOGATE_DIR)
df_sparse = load_results_seed42(SPARSE_DIR)

# ── Build full-model F1 per cancer per gate type ──────────────────────────────
full_6_tuple = tuple(range(6))

soft_full   = {}
nogate_full = {}
sparse_full = {}

for cancer in TARGET_CANCERS:
    # Soft: from ablation_results.pkl
    soft_full[cancer] = ablation_results.get(cancer, {}).get(full_6_tuple, np.nan)

    # No-gate: from JSONs
    sub = df_nogate[(df_nogate['cancer']==cancer) & (df_nogate['combo_name']==full_6)]
    nogate_full[cancer] = sub['test_f1'].mean() if len(sub) > 0 else np.nan

    # Sparse: from JSONs
    sub = df_sparse[(df_sparse['cancer']==cancer) & (df_sparse['combo_name']==full_6)]
    sparse_full[cancer] = sub['test_f1'].mean() if len(sub) > 0 else np.nan

# ── Plot ──────────────────────────────────────────────────────────────────────
cancers_short = [c.replace('TCGA-','') for c in TARGET_CANCERS]
x     = np.arange(len(TARGET_CANCERS))
width = 0.26

fig, ax = plt.subplots(figsize=(11, 5))

for i, (gate_name, data, style) in enumerate([
    ('soft',   soft_full,   GATE_STYLES['soft']),
    ('none',   nogate_full, GATE_STYLES['none']),
    ('sparse', sparse_full, GATE_STYLES['sparse']),
]):
    vals = [data[c] for c in TARGET_CANCERS]
    bars = ax.bar(x + (i-1)*width, vals, width=width,
                  color=style['color'], label=style['label'],
                  edgecolor='white', alpha=0.9)
    for xi, val in zip(x, vals):
        if not np.isnan(val):
            ax.text(xi + (i-1)*width, val + 0.01, f'{val:.3f}',
                    ha='center', fontsize=8.5, color='#1a1a1a')

ax.set_xticks(x)
ax.set_xticklabels(cancers_short, fontsize=12)
ax.set_ylabel('Macro F1 (full 6-modality model)', fontsize=11)
ax.set_title('Full Model Performance by Gate Type\n(Higher = gate handles noise better — seed=42)',
             fontweight='bold', fontsize=13, color=C['navy'])
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_full_model_comparison_complete.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the numbers cleanly
print(f"\n{'Cancer':<12} {'Soft':>8} {'No-gate':>10} {'Sparse':>10}")
print("-"*44)
for cancer in TARGET_CANCERS:
    print(f"{cancer.replace('TCGA-',''):<12} "
          f"{soft_full[cancer]:>8.4f} "
          f"{nogate_full[cancer]:>10.4f} "
          f"{sparse_full[cancer]:>10.4f}")


Cancer           Soft    No-gate     Sparse
--------------------------------------------
BRCA           0.7356     0.6841     0.6706
GBM_LGG        0.8217     0.7639     0.8884
UCEC           0.6104     0.5266     0.6205
THCA           0.7138     0.7557     0.6809
KIRC           0.4704     0.5746     0.4149


In [ ]:
import json
from pathlib import Path

KIRC_rna_files = list(NOGATE_DIR.glob('KIRC_rna_s42_f0.json'))
if KIRC_rna_files:
    with open(KIRC_rna_files[0]) as f:
        print(json.load(f))
else:
    print("File not found")

{'cancer': 'TCGA-KIRC', 'combo': [0], 'combo_name': 'rna', 'seed': 42, 'fold': 0, 'n_folds': 5, 'test_f1': 0.7206524375454766, 'test_acc': 0.7261904761904762, 'best_val_f1': 0.8745819397993311, 'n_train': 274, 'n_val': 59, 'n_test': 84, 'model_type': 'NoGateModel'}


In [ ]:
# Check what the by-N figure is actually pulling for KIRC n=1
import json
import pandas as pd

def load_results_seed42(save_dir):
    rows = []
    for f in save_dir.glob('*.json'):
        with open(f) as fp:
            r = json.load(fp)
        if r['seed'] == 42 and r.get('fold', 0) == 0:
            rows.append(r)
    return pd.DataFrame(rows) if rows else pd.DataFrame()

df_nogate = load_results_seed42(NOGATE_DIR)
df_sparse = load_results_seed42(SPARSE_DIR)

# Check n=1 KIRC for both
for name, df in [('no-gate', df_nogate), ('sparse', df_sparse)]:
    kirc = df[df['cancer']=='TCGA-KIRC']
    kirc['n_mod'] = kirc['combo_name'].apply(lambda x: len(x.split('+')))
    n1 = kirc[kirc['n_mod']==1]
    print(f"\n{name} KIRC n=1 results:")
    print(n1[['combo_name','test_f1']].to_string())


no-gate KIRC n=1 results:
    combo_name   test_f1
292        rna  0.720652
293       meth  0.434478
294        cnv  0.251634
295      mirna  0.541667
296        mut  0.131579
297       clin  0.169210

sparse KIRC n=1 results:
    combo_name   test_f1
292        rna  0.730039
293       meth  0.414069
294        cnv  0.248364
295      mirna  0.441522
296        mut  0.131579
297       clin  0.183025


/tmp/ipykernel_667/3267386222.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kirc['n_mod'] = kirc['combo_name'].apply(lambda x: len(x.split('+')))
/tmp/ipykernel_667/3267386222.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kirc['n_mod'] = kirc['combo_name'].apply(lambda x: len(x.split('+')))


In [ ]:
# ── Fixed KIRC by-N gate comparison figure ────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
full_6 = '+'.join(MODALITY_NAMES)

def best_per_n(df, cancer):
    sub = df[df['cancer'] == cancer].copy()
    sub['n_mod'] = sub['combo_name'].apply(lambda x: len(x.split('+')))
    return sub.groupby('n_mod')['test_f1'].max()

# Soft gate: from ablation_results.pkl (single seed, seed=42 equivalent)
soft_per_n = {}
for n in range(1, 7):
    combos_n = [c for c in ablation_results.get('TCGA-KIRC', {}) if len(c) == n]
    if combos_n:
        soft_per_n[n] = max(ablation_results['TCGA-KIRC'][c] for c in combos_n)

# No-gate and sparse: from JSONs filtered to seed=42 fold=0
df_nogate_42 = load_results_seed42(NOGATE_DIR)
df_sparse_42 = load_results_seed42(SPARSE_DIR)

nogate_per_n = best_per_n(df_nogate_42, 'TCGA-KIRC')
sparse_per_n = best_per_n(df_sparse_42, 'TCGA-KIRC')

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
styles = {
    'Soft gate (sigmoid)':    (soft_per_n,   '#2E86AB', 'o', '-'),
    'No gate (concat)':       (nogate_per_n, '#F59E0B', 's', '--'),
    'Sparse gate (sparsemax)':(sparse_per_n, '#C73E1D', '^', '-.'),
}
for label, (data, color, marker, ls) in styles.items():
    if isinstance(data, dict):
        xs = sorted(data.keys())
        ys = [data[x] for x in xs]
    else:
        xs = data.index.tolist()
        ys = data.values.tolist()
    ax.plot(xs, ys, color=color, linestyle=ls, marker=marker,
            markersize=8, linewidth=2, label=label,
            markerfacecolor='white', markeredgewidth=2)

ax.set_xlabel('Number of Modalities', fontsize=12)
ax.set_ylabel('Best Macro F1', fontsize=12)
ax.set_title('KIRC: Best Performance by Modality Count\nThree Gate Types Compared (seed=42, retrain from scratch)',
             fontweight='bold', fontsize=13, color='#0D1F3C')
ax.set_xticks(range(1, 7))
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_kirc_by_n_gate_comparison_fixed.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig4_kirc_by_n_gate_comparison_fixed.png")

Saved fig4_kirc_by_n_gate_comparison_fixed.png


In [ ]:
# Get soft gate full-6 results for UCEC and THCA from ablation_results.pkl
import pickle

with open(SUB_DIR / 'ablation_results.pkl', 'rb') as f:
    ablation_results = pickle.load(f)

MODALITY_NAMES = ['rna', 'meth', 'cnv', 'mirna', 'mut', 'clin']
full_6_tuple = tuple(range(6))

print("Soft gate full-6 results (from ablation_results.pkl):")
for cancer in ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']:
    val = ablation_results.get(cancer, {}).get(full_6_tuple, None)
    print(f"  {cancer.replace('TCGA-',''):<10} {val:.4f}" if val else f"  {cancer.replace('TCGA-',''):<10} NOT FOUND")

print("\nSoft gate best combo results:")
for cancer in ['TCGA-BRCA', 'TCGA-GBM_LGG', 'TCGA-UCEC', 'TCGA-THCA', 'TCGA-KIRC']:
    best_combo = max(ablation_results.get(cancer, {}),
                     key=ablation_results.get(cancer, {}).get, default=None)
    if best_combo:
        best_f1 = ablation_results[cancer][best_combo]
        best_name = '+'.join(MODALITY_NAMES[i] for i in best_combo)
        print(f"  {cancer.replace('TCGA-',''):<10} {best_f1:.4f}  ({best_name})")

Soft gate full-6 results (from ablation_results.pkl):
  BRCA       0.7356
  GBM_LGG    0.8217
  UCEC       0.6104
  THCA       0.7138
  KIRC       0.4704

Soft gate best combo results:
  BRCA       0.7513  (rna+meth+cnv+mut)
  GBM_LGG    0.8713  (rna+meth+mirna)
  UCEC       0.7634  (rna+meth+cnv+clin)
  THCA       0.8171  (rna+meth+mut)
  KIRC       0.8722  (rna+clin)


In [ ]:
df_sparse_42 = load_results_seed42(SPARSE_DIR)
kirc_sparse = df_sparse_42[df_sparse_42['cancer']=='TCGA-KIRC'].copy()
kirc_sparse['n_mod'] = kirc_sparse['combo_name'].apply(lambda x: len(x.split('+')))

# Best combo at n=5
n5 = kirc_sparse[kirc_sparse['n_mod']==5].sort_values('test_f1', ascending=False).head(3)
print("Best sparse KIRC n=5 combos:")
print(n5[['combo_name','test_f1']].to_string())

Best sparse KIRC n=5 combos:
                 combo_name   test_f1
35    rna+meth+cnv+mut+clin  0.726316
37   rna+cnv+mirna+mut+clin  0.713083
34  rna+meth+cnv+mirna+clin  0.542885


In [ ]:
# ── Top 15 combos for no-gate and sparse gate KIRC (multi-seed) ───────────────
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as scipy_stats

def summarize_kirc(save_dir, gate_name, color):
    """Load all KIRC results, compute mean ± CI per combo, plot top 15."""
    rows = []
    for f in save_dir.glob('KIRC_*.json'):
        with open(f) as fp:
            r = json.load(fp)
        rows.append(r)
    df = pd.DataFrame(rows)

    grp = df.groupby('combo_name')['test_f1']
    summary = grp.agg(['mean','std','count']).reset_index()
    summary.columns = ['combo_name','mean_f1','std_f1','n_runs']
    summary['ci95'] = summary.apply(
        lambda r: scipy_stats.t.ppf(0.975, r['n_runs']-1) * r['std_f1'] / np.sqrt(r['n_runs'])
        if r['n_runs'] > 1 else np.nan, axis=1)
    summary = summary.sort_values('mean_f1', ascending=False)

    full_6 = '+'.join(MODALITY_NAMES)
    full_row = summary[summary['combo_name'] == full_6].iloc[0]

    top15 = summary.head(15).iloc[::-1]

    fig, ax = plt.subplots(figsize=(12, 6))
    colors_bars = [color if r['combo_name'] != full_6 else '#C0392B'
                   for _, r in top15.iterrows()]
    ax.barh(top15['combo_name'], top15['mean_f1'],
            xerr=top15['ci95'], color=colors_bars,
            edgecolor='white', capsize=4, linewidth=0.5)

    for _, row in top15.iterrows():
        ax.text(row['mean_f1'] + row['ci95'] + 0.005,
                top15.index.get_loc(row.name) if False else
                top15['combo_name'].tolist().index(row['combo_name']),
                f"{row['mean_f1']:.3f}±{row['std_f1']:.3f}",
                va='center', fontsize=8.5, color='#0D1F3C')

    ax.axvline(full_row['mean_f1'], color='#C0392B', linestyle='--',
               linewidth=1.5,
               label=f"Full 6-mod: {full_row['mean_f1']:.3f}±{full_row['std_f1']:.3f}")
    ax.set_xlabel('Macro F1 (mean ± 95% CI across 25 runs)', fontsize=12)
    ax.set_title(f'KIRC: Top 15 Modality Combinations — {gate_name}\n5 seeds × 5-fold CV',
                 fontweight='bold', fontsize=13, color='#0D1F3C')
    ax.set_xlim(0, 1.05)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color=color,      label=f'Top combinations'),
        Patch(color='#C0392B',  label='Full 6-modality model'),
    ], loc='lower right', fontsize=10)

    plt.tight_layout()
    fname = FIG_DIR / f'kirc_top15_{gate_name.lower().replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved {fname}")

    print(f"\n{gate_name} — top 5:")
    print(summary.head(5)[['combo_name','mean_f1','std_f1','ci95','n_runs']].to_string(index=False))
    print(f"\nFull-6: mean={full_row['mean_f1']:.4f} std={full_row['std_f1']:.4f}")
    print(f"Delta (best - full-6): {summary.iloc[0]['mean_f1'] - full_row['mean_f1']:+.4f}")

summarize_kirc(NOGATE_DIR, "No Gate (concat)",     "#F59E0B")
summarize_kirc(SPARSE_DIR, "Sparse Gate (sparsemax)", "#C73E1D")

Saved /content/drive/MyDrive/tcga_pancancer/results/attention_comparison_figures/kirc_top15_no_gate_(concat).png

No Gate (concat) — top 5:
   combo_name  mean_f1   std_f1     ci95  n_runs
      rna+mut 0.726609 0.042465 0.017529      25
     rna+clin 0.718936 0.044708 0.018455      25
          rna 0.707352 0.054299 0.022414      25
rna+meth+clin 0.702695 0.049848 0.020576      25
 rna+mut+clin 0.702657 0.045680 0.018856      25

Full-6: mean=0.5443 std=0.0646
Delta (best - full-6): +0.1823
Saved /content/drive/MyDrive/tcga_pancancer/results/attention_comparison_figures/kirc_top15_sparse_gate_(sparsemax).png

Sparse Gate (sparsemax) — top 5:
            combo_name  mean_f1   std_f1     ci95  n_runs
          rna+mut+clin 0.715980 0.035787 0.014772      25
              rna+clin 0.713904 0.031819 0.013134      25
                   rna 0.712774 0.061923 0.025561      25
               rna+mut 0.703893 0.056472 0.023311      25
rna+cnv+mirna+mut+clin 0.661170 0.096862 0.039983      25

